## Merge line data with scatter data to sort them into categories

Datapoints get :

- level: n80, n90, n70, n10, n20, n30, -

- olulisus: p-value

- annotated word count (form, ekilex_tag=location)

- not annotated word count (form, ekilex_tag is null)

- unique lemma count (ekilex_tag=location)


Updated scatter_lines_df3 can be used to later extract info for gpt

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import binom
import copy

In [ ]:
DATABASE = "../drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"

LINE_DATA_TABLE = "lines_class_info3"
LINE_DATA_TABLE2 = "lines_class_info4"

## andmetabelid

In [2]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

In [3]:
# graafiku joonte info
query = f"SELECT * FROM scatter_lines_df3"

lines_df = pd.read_sql(query, conn)


In [4]:
lines_df

,x,y_pos80,log2_x,y_neg80,log2_y_pos80,y_pos90,y_neg90,log2_y_pos90,y_pos70,y_neg70,...,log2_y_pos90_p1,log2_y_pos90_m1,log2_y_pos70_p1,log2_y_pos70_m1,log2_y_pos20_p1,log2_y_pos20_m1,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1
0,1,1,0.000000,0,inf,1,0,inf,0,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
1,2,2,1.000000,0,inf,2,0,inf,0,2,...,NaN,0.000000,0.000000,NaN,0.000000,NaN,0.000000,NaN,NaN,0.000000
2,3,3,1.584963,0,inf,3,0,inf,1,2,...,NaN,1.000000,1.000000,-inf,-1.000000,NaN,-1.000000,NaN,NaN,1.000000
3,4,4,2.000000,0,inf,4,0,inf,1,3,...,NaN,1.584963,0.000000,-inf,-1.584963,NaN,-1.584963,NaN,NaN,1.584963
4,5,5,2.321928,0,inf,5,0,inf,2,3,...,NaN,2.000000,0.584963,-2.000000,-2.000000,NaN,-2.000000,NaN,inf,0.584963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4794,4795,3882,12.227315,913,2.088113,4350,445,3.289138,3304,1491,...,3.292715,3.285568,1.149338,1.146529,-2.084213,-2.088113,-3.282005,-3.289138,-1.145126,-1.147933
4795,4796,3883,12.227616,913,2.088485,4351,445,3.289470,3305,1491,...,3.293047,3.285900,1.149774,1.146966,-2.084585,-2.088485,-3.282337,-3.289470,-1.145563,-1.148370
4796,4797,3884,12.227917,913,2.088856,4352,445,3.289801,3306,1491,...,3.293378,3.286231,1.150211,1.147403,-2.084956,-2.088856,-3.282669,-3.289801,-1.146000,-1.148806
4797,4798,3885,12.228217,913,2.089228,4353,445,3.290133,3306,1492,...,3.293710,3.286563,1.149243,1.146436,-2.085328,-2.089228,-3.283000,-3.290133,-1.145034,-1.147839


In [5]:
# tabel, kus on verb+kääne koos graafiku infoga ja uniq lemma arvuga

query = f"SELECT * FROM verb_case_log_location"

verb_case_log_location = pd.read_sql(query, conn)
verb_case_log_location

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3
1,aasima,,in,0.000000,-1.321928,7,2,1.000000,2-3
2,abielluma,,abl,9.965784,-5.658211,103,2,1.000000,1
3,abielluma,,ad,-2.807355,-0.880761,1182,78,6.285402,1
4,abielluma,,adit,-3.584963,0.378512,23,4,2.000000,1
...,...,...,...,...,...,...,...,...,...
20935,šokeerima,,ad,-3.502500,-0.980371,110,25,4.643856,2-3
20936,šokeerima,,all,9.965784,-2.321928,6,1,0.000000,2-3
20937,šokeerima,,el,-9.965784,-1.584963,16,4,2.000000,2-3
20938,šokeerima,,in,1.584963,-0.536053,49,16,4.000000,2-3


## merge

In [6]:
new_df = pd.merge(verb_case_log_location, lines_df, left_on='unique_lemmas', right_on='x')
new_df

,verb,verb_compound,morph_case,log2_tag,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos90_p1,log2_y_pos90_m1,log2_y_pos70_p1,log2_y_pos70_m1,log2_y_pos20_p1,log2_y_pos20_m1,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
1,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
2,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
3,aerutama,,in,-9.965784,-1.736966,13,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
4,aevastama,,in,9.965784,-2.584963,7,1,0.000000,1,1,...,NaN,-inf,inf,NaN,inf,NaN,inf,NaN,NaN,-inf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-1.777094,22274,1313,10.358651,2-3,1313,...,3.422666,3.394726,1.088271,1.078182,-2.160544,-2.175303,-3.380922,-3.408640,-1.073148,-1.083223
20936,õppima,,in,5.233872,-0.004661,11763,530,9.049849,2-3,530,...,3.614710,3.538420,1.020464,0.995919,-2.251225,-2.289507,-3.501513,-3.576139,-0.983698,-1.008174
20937,ütlema,,ad,-5.370614,0.326842,22466,415,8.696968,2-3,415,...,3.681824,3.581201,0.989583,0.958481,-2.276518,-2.326104,-3.533035,-3.630766,-0.943010,-0.974005
20938,ütlema,,all,0.271387,-1.252183,57846,1131,10.143383,2-3,1131,...,3.446953,3.414108,1.075288,1.063616,-2.175388,-2.192645,-3.397915,-3.430453,-1.057793,-1.069448


## millesesse tsooni punkt kuulub

kas >= 90

<90 ja >=80

<= 70 ja >= 30

<20

<10


In [7]:
def get_level(row):
    if row['log2_tag'] >= row[f"log2_y_pos90"]: #kõrgemal 90 joonest
        return "n90"
    elif row['log2_tag'] >= row[f"log2_y_pos80"] and row['log2_tag'] < row[f"log2_y_pos90"]: #kõrgemal 80 joonest
        return "n80"
    elif row['log2_tag'] >= 0 and row['log2_tag'] <= row[f"log2_y_pos70"]: #madalamal 70 ja kõrgemal 0 joonest
        return "n70"
    elif row['log2_tag'] >= row[f"log2_y_pos30"] and row['log2_tag'] <= 0: #madalamal 0 ja kõrgemal 30 joonest
        return "n30"
    elif row['log2_tag'] <= row[f"log2_y_pos20"] and row['log2_tag'] > row[f"log2_y_pos10"]: #madalamal 20 joonest
        return "n20"
    elif row['log2_tag'] <= row[f"log2_y_pos10"]: #madalamal 10 joonest
        return "n10"
    else:
        return "-"

In [8]:
new_df["level"] = new_df.apply(get_level, axis=1)

## distance, ehk kui kõrgel on punkt joonest



def distance(row):
    if row["level"] == "n90":
        return row['log2_tag']-row[f"log2_y_pos90"]
    if row["level"] == "n80":
        return row['log2_tag']-row[f"log2_y_pos80"]
    if row["level"] == "n10":
        return row['log2_tag']-row[f"log2_y_pos10"]
    if row["level"] == "n20":
        return row['log2_tag']-row[f"log2_y_pos20"]
    if row["level"] == "n70":
        return row[f"log2_y_pos70"]- row['log2_tag']
    if row["level"] == "n30":
        return row[f"log2_y_pos30"]- row['log2_tag']

In [10]:
#new_df["distance"] = new_df.apply(distance, axis=1)

## not_ann_words ja ann_words 

(mitte lemmad vaid spatial_obl tabelist 'form' kus ekilex_tag on kas null või mitte)

In [9]:
query = """
        SELECT verb, verb_compound, morph_case, count(form) as not_ann_words
        FROM spatial_obl
        WHERE ekilex_tag is null
        GROUP BY verb, verb_compound, morph_case;
        """

df_notag = pd.read_sql(query, conn)
df_notag

,verb,verb_compound,morph_case,not_ann_words
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
66037,šveitsima,,in,6
66038,žestikuleerima,,ad,1
66039,žongleerima,,ad,3
66040,žongleerima,,in,2


In [10]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_notag, on=['verb', 'verb_compound', 'morph_case'], how='left')

In [11]:
tags = "('location')"

query = f"""
        SELECT verb, verb_compound, morph_case, count(form) as ann_words
        FROM spatial_obl
        WHERE ekilex_tag IN {tags}
        GROUP BY verb, verb_compound, morph_case;
        """

df_tag = pd.read_sql(query, conn)
df_tag

,verb,verb_compound,morph_case,ann_words
0,21olema,,in,1
1,A. kollama,,ad,1
2,A. tihkama,,abl,1
3,B. kurtma,,in,1
4,B. teadma,,in,1
...,...,...,...,...
25795,šokeerima,,adit,1
25796,šokeerima,,all,1
25797,šokeerima,,ill,1
25798,šokeerima,,in,15


In [12]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## kui palju unique lemmasid ära katab

In [13]:
tags = "('location')"

query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS ann_unique_lemmas
    FROM spatial_obl
    WHERE ekilex_tag in {tags}
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_tag = pd.read_sql(query, conn)
df_ul_tag

,verb,verb_compound,morph_case,ann_unique_lemmas
0,21olema,,in,1
1,A. kollama,,ad,1
2,A. tihkama,,abl,1
3,B. kurtma,,in,1
4,B. teadma,,in,1
...,...,...,...,...
25795,šokeerima,,adit,1
25796,šokeerima,,all,1
25797,šokeerima,,ill,1
25798,šokeerima,,in,11


In [14]:
# Merge with df_log based on verb, verb_compound, morph_case
new_df = new_df.merge(df_ul_tag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## kui palju unique lemmasid on annoteerimata

In [15]:
query = f"""
    SELECT 
        verb, 
        verb_compound,
        morph_case, 
        COUNT(DISTINCT lemma) AS not_ann_unique_lemmas
    FROM spatial_obl
    WHERE ekilex_tag is null
    GROUP BY verb, verb_compound, morph_case
"""

df_ul_ntag = pd.read_sql(query, conn)
df_ul_ntag

,verb,verb_compound,morph_case,not_ann_unique_lemmas
0,0muutuma,,el,1
1,0olema,,ad,1
2,0olema,,el,2
3,0olema,,in,2
4,10halama,,in,1
...,...,...,...,...
66037,šveitsima,,in,6
66038,žestikuleerima,,ad,1
66039,žongleerima,,ad,2
66040,žongleerima,,in,2


In [16]:
new_df = new_df.merge(df_ul_ntag, on=['verb', 'verb_compound', 'morph_case'], how='left')

## olulisus (p-value)


"""
def get_min_success(n=100, p=0.8, kv=0.05):

    #n = 100       # number of trials
    #p = 0.8       # null hypothesis success rate

    # Find smallest k such that P(X ≥ k) < 0.05
    for k in range(n + 1):
        if binom.sf(k - 1, n, p) <= kv: #kv=0.05 95% puhul, 70% puhul peaks olema 0.05 asemel 0.95
            #print(f"Minimum k: {k}")
            #break
            return k
    
    #return np.nan
    return n
"""

def kv_for_datapoint(n, log2_ratio, p=0.8): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x
    # Compute observed successes from log2(y_pos/y_neg)
    ratio = 2 ** log2_ratio
    k_obs = n * ratio / (1 + ratio)
    # Compute probability P(X >= k_obs)
    kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
    return n, k_obs, kv_obs

### Example: line point at (11.16, 2.13) corresponds to 5% line
### Above-point at (11.16, 3.06)
n_line, k_line, kv_line = kv_for_datapoint(1534, 2.163039, p=0.8)
n_obs, k_obs, kv_obs = kv_for_datapoint(1534, 2.236495, p=0.8) # (log2_unique_lemmas, log2_tag, p)

print(f"n (unique lemmas) = {n_line:.0f}")
print(f"Critical line point: k = {k_line:.0f} (y_pos80), kv = {kv_line:.4f}")
print(f"Above point: k = {k_obs:.0f}, kv = {kv_obs:.6f}")

**CDF = left side = P(X ≤ k)**

**SF = right side = P(X > k)**

**SF(k-1) = P(X ≥ k)**

**use CDF for lower tail, SF for upper tail**

In [17]:
mapping_level = {"n80": 0.8, "n90": 0.9, "n70":0.7, "n20": 0.2, "n10":0.1, "n30": 0.3}

def kv_for_datapoint(row): #(log2_unique_lemmas->unique_lemmas, log2_tag, p)
    #n=2**log2_x : log2_x = log2_unique_lemmas
    # n : unique_lemmas
    # log2_ratio : log2_tag
    # p = 0.8 kui n80
    
    if row["level"] != "-":
        n = row["unique_lemmas"]
        log2_ratio = row["log2_tag"]
        p = mapping_level[row["level"]]

        # Compute observed successes from log2(y_pos/y_neg)
        ratio = 2 ** log2_ratio
        k_obs = n * ratio / (1 + ratio)
        # Compute probability P(X >= k_obs), üleval pool joont
        if p>0.5:
            kv_obs = binom.sf(int(round(k_obs)) - 1, int(round(n)), p)
        else: # all pool joont
            kv_obs = binom.cdf(int(round(k_obs)), int(round(n)), p)
        #return n, k_obs, kv_obs
        return round(kv_obs, 5)
    else:
        return "-"

In [18]:
new_df["olulisus"] = new_df.apply(kv_for_datapoint, axis=1)

In [19]:
new_df = new_df.rename(columns={'log2_tag': 'log2_ratio'})

In [20]:
new_df

,verb,verb_compound,morph_case,log2_ratio,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,log2_y_pos10_p1,log2_y_pos10_m1,log2_y_pos30_p1,log2_y_pos30_m1,level,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3,1,...,inf,NaN,NaN,-inf,-,7.0,NaN,NaN,5.0,-
1,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,inf,NaN,NaN,-inf,-,15.0,NaN,NaN,12.0,-
2,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,8.0,NaN,NaN,7.0,-
3,aerutama,,in,-9.965784,-1.736966,13,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,10.0,NaN,NaN,5.0,-
4,aevastama,,in,9.965784,-2.584963,7,1,0.000000,1,1,...,inf,NaN,NaN,-inf,-,6.0,1.0,1.0,6.0,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-1.777094,22274,1313,10.358651,2-3,1313,...,-3.380922,-3.408640,-1.073148,-1.083223,-,17243.0,1022.0,331.0,2522.0,-
20936,õppima,,in,5.233872,-0.004661,11763,530,9.049849,2-3,530,...,-3.501513,-3.576139,-0.983698,-1.008174,n90,5891.0,5720.0,463.0,985.0,0.0
20937,ütlema,,ad,-5.370614,0.326842,22466,415,8.696968,2-3,415,...,-3.533035,-3.630766,-0.943010,-0.974005,n10,9966.0,295.0,104.0,1162.0,0.0
20938,ütlema,,all,0.271387,-1.252183,57846,1131,10.143383,2-3,1131,...,-3.397915,-3.430453,-1.057793,-1.069448,n70,40742.0,9354.0,189.0,2337.0,1.0


In [21]:
new_df.to_sql(LINE_DATA_TABLE, conn, if_exists="replace", index=False)

20940

## Juurde, kui palju oli location tag arv ja mitte location tag arv

In [22]:
query = f"""SELECT tbl1.verb, tbl1.verb_compound, tbl1.morph_case, 
        log2_ratio, log2_annotation,tbl1.verb_case_count, unique_lemmas, 
        log2_unique_lemmas,synset_count, x, y_pos80, log2_x, y_neg80, log2_y_pos80,
       y_pos90, y_neg90, log2_y_pos90, y_pos70, y_neg70,log2_y_pos70, y_pos30, y_neg30, log2_y_pos30, y_pos20,
       y_neg20, log2_y_pos20, y_pos10, y_neg10, log2_y_pos10,
       level, not_ann_words, ann_words, ann_unique_lemmas,
       not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated,
       verb_case_counts_location.verb_case_count as loc_verb_case_count

            FROM {LINE_DATA_TABLE} as tbl1
            left join 
            verb_case_counts_location 
            on 
            tbl1.verb = verb_case_counts_location.verb and
            tbl1.verb_compound = verb_case_counts_location.verb_compound and
            tbl1.morph_case = verb_case_counts_location.morph_case
            """

df = pd.read_sql(query, conn)
df

,verb,verb_compound,morph_case,log2_ratio,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,loc_verb_case_count
0,aasima,,ad,-9.965784,-2.807355,8,1,0.000000,2-3,1,...,7.0,NaN,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-3.906891,16,1,0.000000,2-3,1,...,15.0,NaN,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-3.000000,9,1,0.000000,1,1,...,8.0,NaN,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-1.736966,13,1,0.000000,1,1,...,10.0,NaN,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-2.584963,7,1,0.000000,1,1,...,6.0,1.0,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-1.777094,22274,1313,10.358651,2-3,1313,...,17243.0,1022.0,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,-0.004661,11763,530,9.049849,2-3,530,...,5891.0,5720.0,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,0.326842,22466,415,8.696968,2-3,415,...,9966.0,295.0,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,-1.252183,57846,1131,10.143383,2-3,1131,...,40742.0,9354.0,189.0,2337.0,1.0,9354,7750,17104,40742,57846


In [ ]:
# verb_case_count peaks olema sama, mis loc_verb_case_count, ehk võib võtta ühe
# ann_words peaks olema sama, mis my_tag (NaN vs 0 ka)
# not_ann_words peaks olema sama, mis not_annotated (Nan vs 0 ka)


In [23]:
df[df["verb"]=="käima"]

,verb,verb_compound,morph_case,log2_ratio,log2_annotation,verb_case_count,unique_lemmas,log2_unique_lemmas,synset_count,x,...,not_ann_words,ann_words,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,loc_verb_case_count
1037,käima,kallal,ad,-9.965784,-1.807355,9,1,0.000000,not in Estonian Wordnet,1,...,7.0,NaN,NaN,7.0,-,0,2,2,7,9
1038,käima,kannul,ad,-9.965784,-2.807355,8,1,0.000000,not in Estonian Wordnet,1,...,7.0,NaN,NaN,4.0,-,0,1,1,7,8
1039,käima,kinni,in,9.965784,-2.321928,6,1,0.000000,not in Estonian Wordnet,1,...,5.0,1.0,1.0,5.0,-,1,0,1,5,6
1040,käima,ringi,all,-9.965784,-2.321928,6,1,0.000000,not in Estonian Wordnet,1,...,5.0,NaN,NaN,4.0,-,0,1,1,5,6
1041,käima,sisse,el,9.965784,-4.392317,22,1,0.000000,not in Estonian Wordnet,1,...,21.0,1.0,1.0,14.0,-,1,0,1,21,22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20593,käima,,adit,1.354533,-2.620257,1780,124,6.954196,>3,124,...,1531.0,179.0,80.0,421.0,-,179,70,249,1531,1780
20598,käima,,all,-1.603578,-2.059450,4822,461,8.848623,>3,461,...,3889.0,231.0,121.0,856.0,-,231,702,933,3889,4822
20599,käima,,el,-0.146037,-1.403821,8003,837,9.709084,>3,837,...,5808.0,1042.0,397.0,1979.0,1.0,1042,1153,2195,5808,8003
20600,käima,,in,3.155219,-0.306129,49527,2220,11.116344,>3,2220,...,27381.0,19911.0,1903.0,4636.0,0.0,19911,2235,22146,27381,49527


In [24]:
# log2_ratio = np.log2((my_tag/annotated)/(other_tags/annotated))
np.log2((167/295)/(128/295))

0.38370429247405224

## salvestada andmebaasi

In [25]:
#new_df.to_sql("lines_class_info3", conn, if_exists="replace", index=False)
df.to_sql(LINE_DATA_TABLE2, conn, if_exists="replace", index=False)

20940

## uuesti sisse lugemine

In [26]:
# log2_unique_lemmas on sama, mis log2_x
# log2_tag on andmepunkti y 

query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, unique_lemmas, level, 
            ann_unique_lemmas, not_ann_unique_lemmas, olulisus,
            my_tag, other_tags, annotated, not_annotated
            FROM {LINE_DATA_TABLE2}
            """

df = pd.read_sql(query, conn)

In [27]:
df

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
0,aasima,,ad,-9.965784,1,-,NaN,5.0,-,0,1,1,7
1,abistama,,all,-9.965784,1,-,NaN,12.0,-,0,1,1,15
2,aeglustama,,all,-9.965784,1,-,NaN,7.0,-,0,1,1,8
3,aerutama,,in,-9.965784,1,-,NaN,5.0,-,0,3,3,10
4,aevastama,,in,9.965784,1,-,1.0,6.0,-,1,0,1,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,1313,-,331.0,2522.0,-,1022,4009,5031,17243
20936,õppima,,in,5.233872,530,n90,463.0,985.0,0.0,5720,152,5872,5891
20937,ütlema,,ad,-5.370614,415,n10,104.0,1162.0,0.0,295,12205,12500,9966
20938,ütlema,,all,0.271387,1131,n70,189.0,2337.0,1.0,9354,7750,17104,40742


In [28]:
df2 = df[df["level"]!= "-"]

In [29]:
df3 = df2[df2["level"]=="n80"]

In [30]:
df3 = df3.sort_values(["olulisus"])

In [31]:
df3

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
19889,peatuma,,in,3.551257,282,n80,257.0,337.0,0.0,973,83,1056,858
19994,möllama,,in,3.921070,196,n80,177.0,261.0,0.0,409,27,436,495
20637,leiduma,,in,3.086569,518,n80,415.0,1689.0,0.0,1614,190,1804,4760
20673,naasma,,el,3.210249,287,n80,236.0,222.0,0.0,907,98,1005,439
19859,süttima,,in,3.879146,199,n80,177.0,233.0,0.0,515,35,550,661
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18469,nappima,,in,3.496426,140,n80,114.0,198.0,6.0e-05,316,28,344,382
18887,sõitma,edasi,el,5.741467,55,n80,53.0,26.0,7.0e-05,107,2,109,30
20015,teatama,,ill,5.235216,54,n80,51.0,82.0,8.0e-05,113,3,116,640
20194,ootama,,ill,3.882643,108,n80,93.0,132.0,8.0e-05,236,16,252,231


In [32]:
df3[df3["verb"]=="jääma"]

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
15771,jääma,maha,ill,6.321928,37,n80,36.0,30.0,0.00026,80,1,81,33
20275,jääma,,ill,2.540930,645,n80,496.0,1330.0,0.00033,2904,499,3403,8637
18356,jääma,edasi,ill,9.965784,23,n80,23.0,41.0,0.0059,40,0,40,89
17303,jääma,kinni,adit,9.965784,22,n80,22.0,104.0,0.00738,29,0,29,501
17618,jääma,alles,ill,4.321928,31,n80,28.0,88.0,0.00867,60,3,63,113


In [33]:
df3[df3["verb"]=="viima"]

,verb,verb_compound,morph_case,log2_ratio,unique_lemmas,level,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated
17685,viima,kaasa,el,6.209453,31,n80,30.0,42.0,0.00099,74,1,75,67
16293,viima,edasi,ill,9.965784,25,n80,25.0,17.0,0.00378,33,0,33,19
20851,viima,,adit,2.422385,332,n80,255.0,596.0,0.0257,1903,355,2258,3992


In [6]:
conn.close()